# Cutoff Selection

Choosing the seven forecast origins the whole evaluation rests on: **4 event-driven
and 3 quiet**.

**Target series:** CME Crude Palm Oil futures (Yahoo `CPO=F`), weekly, aggregated by
**median** rather than Friday close. MPOB — the physical Malaysian price, and the
cleaner source on every measured axis — was evaluated first but Vector has not
approved it for this project. See [`DATA.md`](DATA.md) for the full three-way
comparison and the roll-mitigation analysis behind the median choice.

The point of the event/quiet split is to separate two questions. On event cutoffs,
does reading news let an agent anticipate a shock a statistical baseline cannot see?
On quiet cutoffs, does the agent *avoid damaging* a forecast when there is nothing to
react to? A method that only wins on shocks and loses on calm weeks is not useful.

Everything here is derived from data. Where the search did not produce a clean
result, that is stated rather than smoothed over — see Section 5.


---
## 1. Setup


In [1]:
from __future__ import annotations

import itertools
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "implementations"))
load_dotenv(ROOT / ".env")

from cpo.data import PALM_OIL_WEEKLY_SERIES_ID, build_palm_oil_futures_service
from cpo.plots import DEFAULT_CUTOFFS, HORIZONS_WEEKS, plot_cutoff_windows, plot_price_history


svc = build_palm_oil_futures_service(cache_dir=ROOT / "data" / "yfinance")
as_of = datetime.now(tz=timezone.utc).replace(tzinfo=None)

weekly = svc.get_series(PALM_OIL_WEEKLY_SERIES_ID, as_of=as_of).set_index("timestamp")["value"]
returns = weekly.pct_change() * 100

news = pd.read_csv(ROOT / "implementations" / "cpo" / "palm_articles_daily.csv")
news["date"] = pd.to_datetime(news["date"], errors="coerce")
news["article_count"] = pd.to_numeric(news["article_count"], errors="coerce")
n_bad = news["date"].isna().sum()
news = news.dropna(subset=["date", "article_count"])
news_weekly = news.set_index("date")["article_count"].resample("W-FRI").sum()

MAX_HORIZON = max(HORIZONS_WEEKS)
print(f"{len(weekly)} weekly prices, {weekly.index.min():%Y-%m-%d} -> {weekly.index.max():%Y-%m-%d}")
print(f"horizons: {HORIZONS_WEEKS} weeks (longest {MAX_HORIZON})")
print(f"dropped {n_bad} malformed rows from the news CSV (embedded commas/newlines in the text column)")

815 weekly prices, 2010-05-28 -> 2026-08-07
horizons: [1, 2, 4, 8, 13] weeks (longest 13)
dropped 18 malformed rows from the news CSV (embedded commas/newlines in the text column)


---
## 2. The four constraints

| # | Constraint | Why |
|---|---|---|
| 1 | Cutoff ≥ 2024-02 | GDELT news starts 2024-01; leave 4 weeks of prior context |
| 2 | Every horizon resolves | A 13-week horizon needs 13 weeks of realised prices after the cutoff |
| 3 | ≥ 100 articles in the prior 8 weeks | An agent needs news to reason over, or the comparison is empty |
| 4 | ≥ 10 weeks between all seven cutoffs | Non-overlapping windows keep the seven scores independent |

Constraint 3 matters more than it looks — GDELT coverage is uneven, and a cutoff in a
sparse stretch would test nothing.


In [2]:
def summarise(cutoff: pd.Timestamp) -> dict | None:
    """Return forward-looking stats for a candidate cutoff, or None if it cannot resolve."""
    targets = [cutoff + pd.Timedelta(weeks=h) for h in HORIZONS_WEEKS]
    if any(t not in weekly.index for t in targets):
        return None
    forward = returns.loc[cutoff + pd.Timedelta(weeks=1) : cutoff + pd.Timedelta(weeks=MAX_HORIZON)]
    return {
        "cutoff": cutoff,
        "price": float(weekly[cutoff]),
        "max_move": float(forward.abs().max()),
        "total_13wk": float(weekly[targets[-1]] / weekly[cutoff] - 1) * 100,
        "news_8wk": float(news_weekly.loc[cutoff - pd.Timedelta(weeks=8) : cutoff].sum()),
    }


pool = pd.DataFrame(
    [s for c in weekly.index if c >= pd.Timestamp("2024-02-01") and (s := summarise(c)) and s["news_8wk"] >= 100]
)
print(f"weeks in the GDELT window        : {(weekly.index >= '2024-02-01').sum()}")
print(f"...that resolve at every horizon and clear the news floor: {len(pool)}")
print(f"latest usable cutoff             : {pool.cutoff.max():%Y-%m-%d}")

weeks in the GDELT window        : 132
...that resolve at every horizon and clear the news floor: 79
latest usable cutoff             : 2026-05-08


---
## 3. Searching for a spaced, well-separated set

Take the 20 largest-move candidates, and search every combination of 4 for one where
all four sit ≥10 weeks apart. A greedy pick (always take the next-largest move) can
miss a valid combination that exists further down the ranking — exhaustive search
does not.


In [3]:
def spaced(dates: list[pd.Timestamp], min_gap_days: int = 70) -> bool:
    d = sorted(dates)
    return all((d[i + 1] - d[i]).days >= min_gap_days for i in range(len(d) - 1))


top20 = pool.sort_values("max_move", ascending=False).head(20)
n_spaced_10wk = sum(1 for combo in itertools.combinations(top20.itertuples(), 4) if spaced([x.cutoff for x in combo]))
print(f"4-of-20 combinations that are mutually >=10 weeks apart: {n_spaced_10wk}")

for gap_weeks in (10, 8, 6):
    n = sum(
        1 for combo in itertools.combinations(top20.itertuples(), 4) if spaced([x.cutoff for x in combo], gap_weeks * 7)
    )
    print(f"  at >={gap_weeks} weeks apart: {n} valid combinations")

4-of-20 combinations that are mutually >=10 weeks apart: 0
  at >=10 weeks apart: 0 valid combinations
  at >=8 weeks apart: 8 valid combinations
  at >=6 weeks apart: 112 valid combinations


**No fully independent 4-event combination exists among the 20 largest moves in this
window** — the large moves in 2024–2026 cluster too closely in time to pull four of
them 10 weeks apart from each other. Relaxing the gap to 8 or 6 weeks makes
combinations available but reintroduces window overlap, which is the property
independence exists to prevent.

**Decision: keep full 10-week independence, accept a weaker event/quiet separation.**
Independence protects the validity of treating seven scores as seven separate data
points; separation strength is a nice-to-have on top of that. Trading it away would
make the whole cutoff set harder to defend, not easier.


---
## 4. The selected seven

Built by fixing four mutually-spaced events from the ranked list, then filling three
quiet slots from the calmest remaining candidates that stay ≥10 weeks from every
chosen cutoff — events and quiet alike.


In [4]:
EVENT_TARGETS = ["2024-04-19", "2024-10-25", "2025-03-28", "2025-06-06"]
events = pool[pool.cutoff.isin(pd.Timestamp(d) for d in EVENT_TARGETS)].copy()
events["kind"] = "event"

quiet_candidates = pool[~pool.cutoff.isin(events.cutoff)].sort_values("max_move")
quiet_dates: list[pd.Timestamp] = []
for row in quiet_candidates.itertuples():
    if all(abs((row.cutoff - c).days) >= 70 for c in list(events.cutoff) + quiet_dates):
        quiet_dates.append(row.cutoff)
        if len(quiet_dates) == 3:
            break

quiet = pool[pool.cutoff.isin(quiet_dates)].copy()
quiet["kind"] = "quiet"

selected = pd.concat([events, quiet]).sort_values("cutoff").reset_index(drop=True)
selected["cutoff_str"] = selected.cutoff.dt.strftime("%Y-%m-%d")
selected[["cutoff_str", "kind", "price", "max_move", "total_13wk", "news_8wk"]].round(1)

,cutoff_str,kind,price,max_move,total_13wk,news_8wk
0,2024-04-19,event,869.8,6.2,-2.9,226.0
1,2024-06-28,quiet,832.0,4.3,10.6,252.0
2,2024-10-25,event,1008.2,9.9,-5.0,161.0
3,2025-01-03,quiet,1015.1,5.7,-2.3,251.0
4,2025-03-28,event,996.0,5.8,-5.7,163.0
5,2025-06-06,event,924.5,3.2,14.2,146.0
6,2026-04-24,quiet,1157.5,1.8,-2.8,140.0


---
## 5. Does the separation actually hold?

State this plainly rather than only report the group averages.


In [5]:
gaps = selected.cutoff.diff().dt.days.dropna()
print(
    f"closest two cutoffs: {int(gaps.min())} days apart ({int(gaps.min()) // 7} weeks) "
    f"-- windows are {'independent' if gaps.min() >= 70 else 'OVERLAPPING'}"
)

ev_moves, qt_moves = events.max_move, quiet.max_move
print(f"\nevent windows -- mean max move {ev_moves.mean():.2f}%  (weakest: {ev_moves.min():.2f}%)")
print(f"quiet windows -- mean max move {qt_moves.mean():.2f}%  (strongest: {qt_moves.max():.2f}%)")
print(f"separation (group means): {ev_moves.mean() / qt_moves.mean():.2f}x")

if qt_moves.max() > ev_moves.min():
    bad_q = quiet.loc[qt_moves.idxmax()]
    bad_e = events.loc[ev_moves.idxmin()]
    print(
        f"\n>> NOT cleanly ordered: quiet {bad_q.cutoff:%Y-%m-%d} moves {bad_q.max_move:.2f}%, "
        f"more than event {bad_e.cutoff:%Y-%m-%d} at {bad_e.max_move:.2f}%."
    )

closest two cutoffs: 70 days apart (10 weeks) -- windows are independent

event windows -- mean max move 6.30%  (weakest: 3.23%)
quiet windows -- mean max move 3.97%  (strongest: 5.73%)
separation (group means): 1.59x

>> NOT cleanly ordered: quiet 2025-01-03 moves 5.73%, more than event 2025-06-06 at 3.23%.


The group means separate (event mean well above quiet mean), but **one pair does
not**: `2025-01-03` is labelled quiet yet moves more than the weakest event,
`2025-06-06`. This is a real property of this window under the 10-week-independence
constraint, not a labelling mistake — a stronger `2025-06-06` was not available
without breaking independence with another chosen cutoff (see Section 3).

This is materially weaker than the analogous split we found when MPOB was still the
candidate target (2.2x, cleanly ordered). The reason is structural, not a search
failure: `CPO=F`'s ordinary-week volatility is lower than MPOB's (1.13% vs 2.06%
mean |move| in non-roll weeks — see `DATA.md`), so its moves compress toward the
middle and are harder to separate into two clean bands.


---
## 6. Visual check

The chart is the audit. If an orange band looks flat, or a blue band contains a
cliff, the label is wrong and the cutoff should be swapped — but per Section 5,
expect the labels to describe a *tendency*, not a hard separation.


In [6]:
plot_cutoff_windows(
    svc.get_series(PALM_OIL_WEEKLY_SERIES_ID, as_of=as_of),
    cutoffs=DEFAULT_CUTOFFS,
    horizons=13,
)

In [7]:
plot_price_history(
    svc.get_series(PALM_OIL_WEEKLY_SERIES_ID, as_of=as_of),
    cutoffs=DEFAULT_CUTOFFS,
    start="2023-06-01",
    title="CPO=F weekly (median), with the seven cutoffs",
    units="USD per tonne",
    currency="$",
    show_blackouts=False,
)

---
## 7. Committed selection

Frozen in `cpo.plots.DEFAULT_CUTOFFS`, so the specs, baselines, and agent
evaluation all read the same list rather than re-deriving it.


In [8]:
pd.DataFrame(
    [{"cutoff": c.date, "kind": c.kind, "why": c.label} for c in sorted(DEFAULT_CUTOFFS, key=lambda c: c.date)]
)

,cutoff,kind,why
0,2024-04-19,event,-6.3% two weeks out; -2.9% over 13 weeks
1,2024-06-28,quiet,max weekly move ahead 4.3%; +10.6% over 13 weeks
2,2024-10-25,event,+10.0% two weeks out; -5.0% over 13 weeks
3,2025-01-03,quiet,max weekly move ahead 5.7% -- exceeds one even...
4,2025-03-28,event,-5.8% two weeks out; -5.7% over 13 weeks
5,2025-06-06,event,+3.2% two weeks out; +14.2% over 13 weeks -- w...
6,2026-04-24,quiet,"calmest window: max 1.9%, -2.8% over 13 weeks"


---
## 8. What this does and does not establish

**Established:** seven origins on a complete weekly grid, resolvable at every
horizon, fully independent (no two forecast windows overlap), each with sufficient
news coverage to give an agent something to reason over.

**Limitations to state in any writeup:**

- **The event/quiet separation is weak and not cleanly ordered.** Group means differ
  1.6x, but the quietest-labelled cutoff (2025-01-03, 5.7%) exceeds the weakest
  event (2025-06-06, 3.2%). This was the best achievable while keeping full
  10-week independence -- see Sections 3 and 5.
- **The target itself carries residual noise.** `CPO=F`'s weekly median reduces the
  monthly contract-roll artifact from a 2.9x to a 1.3x volatility ratio (full
  history) versus MPOB's clean ~1.0x -- better, not solved. See `DATA.md`.
- **Seven origins is a small sample.** Five horizons each gives 35 scored points.
  Mean CRPS differences between close predictors will not be significant. These
  cutoffs are the narrative set; a denser weekly backtest should decide which model
  is actually better.
- **Events were chosen with hindsight.** We know which weeks moved. Fine for a
  controlled comparison, not a live forecasting record.
- **18 of 734 news rows (2.5%) were malformed** and dropped -- likely unescaped
  characters in the source `texts` field. Worth flagging to Jyotsna.
- **MPOB was not used** despite being the cleaner source on every measured axis,
  because Vector has not approved it. If that changes, `build_mpob_service()` is
  already in `data.py` and this notebook's structure carries over directly.

**Next:** a naive last-value baseline across these seven, then Prophet and the
Darts models, then the agent.
